# 05 — Logit Distillation (KL + Temperature)

This notebook is the **first distillation experiment** in the course.

Here is where it sits in the pipeline:

> **pretrain teacher** → **inject facts** → **baseline student** → **logit distillation ← you are here** → **alpha mixing**

Notebook 04 established the **control condition**: a small student trained for 400
steps on hard labels achieved **60.0% cloze**. The architecture, seed, and step
budget were all fixed so that any improvement in this notebook can only come from one
thing: a better training **signal**.

That signal is **knowledge distillation**. Instead of training on one-hot targets
(one correct character, all others zero), the student now trains on the **teacher's
full output distribution** — a probability vector over every character in the
vocabulary. This distribution carries far more information than a single spike.

## Key jargon defined here

| Term | Meaning |
|---|---|
| **Knowledge distillation** | A training technique in which a small *student* model learns to imitate a large *teacher* model's output distribution, rather than (or in addition to) the ground-truth one-hot labels. |
| **Soft labels** | The teacher's output probabilities over all tokens, as opposed to *hard labels* which assign probability 1 to a single correct token and 0 to all others. |
| **Logits** | The raw, unnormalised scores that a neural network produces before applying `softmax`. The teacher's logits are passed to `softmax` to get the soft-label distribution. |
| **Softmax** | The function that converts a vector of logits into a probability distribution: `softmax(z)ᵢ = exp(zᵢ) / Σⱼ exp(zⱼ)`. All values are positive and sum to 1. |
| **Dark knowledge** | The information encoded in the teacher's *non-answer* probabilities. Even when the teacher assigns near-zero probability to a wrong character, the *relative* probabilities among the wrong characters carry structural information about which alternatives are plausible — information that hard labels discard entirely. |
| **KL divergence** | Kullback-Leibler divergence — a measure of how far one probability distribution is from another. `kd_loss` in this notebook is the KL divergence between the teacher's distribution and the student's distribution. |
| **Temperature** | A scalar applied to logits before `softmax` that controls how *peaked* or *spread out* the resulting distribution is. Higher temperature → flatter distribution (more uncertainty); lower temperature → sharper distribution (more confident). |

## What you will see

- The frozen, fact-injected teacher loaded from checkpoint.
- The student trained for 400 steps with `alpha=1.0` (pure KD — no hard-label loss at all).
- Cloze printed at step 0 and step 300.
- The final result: **67.5% cloze**, up from the 60.0% hard-label baseline.
- An `assert` that the distilled student must meet or exceed the baseline.

In [1]:
import os, sys
while not os.path.exists("requirements.txt"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir(parent)
if os.getcwd() not in sys.path:           # so `import model`/`facts` work as a script
    sys.path.insert(0, os.getcwd())

import json
import dataclasses
import torch
from model import GPT, GPTConfig, make_student
from distill import train_distill
from facts import build_factset, render_training_corpus, encode, evaluate_cloze

device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))

def update_metrics(key, value, path="assets/distill_metrics.json"):
    os.makedirs("assets", exist_ok=True)
    m = json.load(open(path)) if os.path.exists(path) else {}
    m[key] = value
    json.dump(m, open(path, "w"), indent=2)
    return m

## Load the (frozen) injected teacher

The teacher is the large GPT trained in notebook 02 and then fact-injected in
notebook 03. It achieves ~97.5% cloze on the held-out probe — near-perfect recall
of the invented facts.

Two things come from the checkpoint:

1. **The shared vocabulary** (`stoi`, `itos`) — the character-level tokenizer that
   maps each character to an integer index. The student *must* use the same vocabulary.
   Distillation works by aligning the student's output distribution with the teacher's:
   if index 42 means `'e'` for the teacher but `'k'` for the student, every soft label
   is meaningless noise.
2. **The teacher model weights** — loaded in `eval()` mode and never updated. The
   teacher is frozen throughout; gradient flow stops at its parameters.

In [2]:
ckpt = torch.load("checkpoints/teacher_injected.pt", weights_only=True)
stoi, itos = ckpt["stoi"], ckpt["itos"]
vocab_size = len(stoi)
teacher = GPT(GPTConfig(**ckpt["config"])).to(device)
teacher.load_state_dict(ckpt["model_state"])
teacher.eval()
facts = build_factset()

## Fact corpus + batching (same as the baseline)

Everything here is identical to notebook 04 — the same corpus, the same `block_size`,
the same `get_batch` function. This is intentional: we are holding every variable
constant so the only difference between this notebook and the baseline is the loss
function.

- `render_training_corpus(facts, repeats=40)` renders the fact-set using
  `templates[1:]` only. The held-out probe (`templates[0]`) never appears in training
  text, so a correct cloze score later proves genuine recall.
- `get_batch` samples random windows of `block_size=256` characters and returns
  `(x, y)` pairs. In the baseline, `y` was used directly as hard-label targets for
  cross-entropy. Here, `y` is passed to `train_distill` which ignores it in favour
  of the teacher's soft output — but the batching mechanism is unchanged.

In [3]:
block_size = 256
corpus = render_training_corpus(facts, repeats=40)
data = torch.tensor(encode(corpus, stoi), dtype=torch.long)

def get_batch(batch_size=32):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

## Distill (pure KD: alpha=1.0) — identical seed/config/steps to the baseline

### What the KL-divergence loss measures

`train_distill` replaces the standard cross-entropy loss with the **KL divergence**
between the teacher's output distribution and the student's output distribution.
In plain language: at every character position in the batch, the teacher computes a
probability vector over the vocabulary (a soft label), and the student is penalised
in proportion to how far its own probability vector is from the teacher's.

If the teacher assigns 90% probability to `'e'`, 5% to `'a'`, and splits the remaining
5% across other characters, the KL loss pushes the student to reproduce that entire
distribution — not just to assign high probability to `'e'`. The student learns the
teacher's *confidence* and its *uncertainty* simultaneously. Hard-label cross-entropy
only teaches the correct answer; KL distillation also teaches "which wrong answers
are nearly right".

This is the essence of **dark knowledge**: even the near-zero probabilities carry
information. Two characters that the teacher rates at 0.01% and 0.001% respectively
are telling the student something about the local structure of the fact — information
that a one-hot target vector throws away.

### Why `alpha=1.0`?

`alpha` is the weight on the KL (distillation) loss. `alpha=1.0` means **pure
distillation** — the hard-label cross-entropy loss is not used at all. This is the
strongest possible test of the soft-label signal: the student must learn entirely
from the teacher's distribution, with no direct supervision from the ground-truth
corpus targets. Notebook 06 will explore mixing `alpha < 1.0` to blend both signals.

### Temperature: what the textbook says vs. what this project measured

**Textbook KD** (Hinton et al., 2015) recommends raising the temperature when
distilling. The intuition: at temperature 1, the teacher's distribution may be
nearly one-hot (very confident), which collapses the soft label toward a hard label
and destroys the dark-knowledge signal. Raising temperature to T=2 or higher
*softens* the distribution, spreading probability mass across near-correct alternatives
and amplifying the relative differences among the wrong answers.

**This project measured the opposite result for our task.** At T=2, pure KD
*underperformed* hard-label training badly. The reason is specific to crisp, discrete
factual recall: the teacher's distribution is already well-calibrated at T=1 — the
correct answer token really does warrant ~90%+ probability, and the small residual
mass on other tokens already carries the dark-knowledge signal. Softening further
with T=2 dilutes the correct-answer signal without recovering additional useful
structure, because the "alternatives" here are truly wrong (no partially-correct
answer exists for a single invented proper noun). The result is a noisier gradient
signal and slower convergence.

**The lesson**: temperature tuning is task-dependent. For ambiguous or paraphrase-rich
targets, high temperature helps. For crisp factual recall with a confident teacher,
T=1 is optimal or even slightly better than any T > 1. Always measure; do not assume
the textbook recommendation transfers.

### The three locked-in constraints (same as the baseline)

The comparison is only meaningful because three things are held constant:

1. **Same student architecture** — `make_student(vocab_size, block_size=256)` produces
   the same small GPT every time.
2. **Same random seed** — `torch.manual_seed(0)` gives the student the same initial
   weights as in notebook 04.
3. **Same step budget** — `STEPS=400` gradient steps, no more.

Any improvement in cloze score is therefore due to the distillation signal alone.

In [4]:
STEPS = 400
torch.manual_seed(0)
student = GPT(make_student(vocab_size, block_size=block_size)).to(device)
train_distill(student, teacher, get_batch, steps=STEPS, lr=3e-4, device=device,
              alpha=1.0, temperature=1.0,
              eval_fn=lambda: f"cloze {evaluate_cloze(student, stoi, itos, facts, device):.1%}",
              log_every=300)

step    0 | loss 4.106 | cloze 0.0%


step  300 | loss 0.204 | cloze 17.5%


step  399 | loss 0.143 | cloze 67.5%


GPT(
  (tok_emb): Embedding(65, 128)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-3): 4 x Block(
      (attn_norm): RMSNorm()
      (attn): CausalSelfAttention(
        (q_proj): Linear(in_features=128, out_features=128, bias=False)
        (k_proj): Linear(in_features=128, out_features=64, bias=False)
        (v_proj): Linear(in_features=128, out_features=64, bias=False)
        (o_proj): Linear(in_features=128, out_features=128, bias=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (mlp_norm): RMSNorm()
      (mlp): SwiGLU(
        (w1): Linear(in_features=128, out_features=384, bias=False)
        (w3): Linear(in_features=128, out_features=384, bias=False)
        (w2): Linear(in_features=384, out_features=128, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=128, out_features=65, bias=False)
)

## Did it beat the baseline?

`evaluate_cloze` feeds the held-out probe prompt for each fact to the student and
checks whether greedy decoding produces the exact correct answer. The result is the
fraction of facts answered correctly.

### The expected result

The logit-distilled student reaches **67.5% cloze** at 400 steps — up from the
60.0% hard-label baseline. Both models used the same architecture, the same seed,
and the same step budget. The 7.5 percentage-point gain is attributable entirely to
the richer training signal: the soft distribution the teacher provides per position
is more informative than the one-hot spike the corpus provides.

This is **sample efficiency** in action: the student extracts more knowledge per
gradient step when learning from a teacher's distribution than when learning from
raw text targets alone.

### The assert

`assert kd_cloze >= baseline` is a hard requirement, not just a hoped-for outcome.
If the distilled student underperforms the baseline, something is broken — the
teacher checkpoint is wrong, the `alpha`/`temperature` settings were accidentally
changed, or the random seed is mismatched. The assert enforces that distillation is
never worse than doing nothing.

### Summary of committed numbers for this run

| Training condition | Cloze (400 steps) |
|---|---|
| Hard-label baseline (notebook 04) | 60.0% |
| Logit-KD, alpha=1.0, T=1.0 (this notebook) | 67.5% |
| Teacher ceiling | ~97.5% |

The student checkpoint is saved to `checkpoints/student_kd.pt` for comparison in
later notebooks. Notebook 06 will explore whether mixing hard labels back in
(`alpha < 1.0`) can close more of the gap to the teacher ceiling.

In [5]:
kd_cloze = evaluate_cloze(student, stoi, itos, facts, device)
metrics = json.load(open("assets/distill_metrics.json"))
baseline = metrics["baseline_hard_label"]
print(f"baseline {baseline:.1%}  ->  logit-KD {kd_cloze:.1%}")
assert kd_cloze >= baseline, "logit distillation should not underperform hard labels"
update_metrics("logit_kd", kd_cloze)

torch.save({"model_state": student.state_dict(),
            "config": dataclasses.asdict(student.config),
            "stoi": stoi, "itos": itos}, "checkpoints/student_kd.pt")
print("Saved checkpoints/student_kd.pt. Continue to 06_alpha_mixing.")

baseline 60.0%  ->  logit-KD 67.5%
Saved checkpoints/student_kd.pt. Continue to 06_alpha_mixing.
